# 1. Setup & Imports

In [2]:
import os, sys, json, torch, penman, numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader


print(f"Pytorch Version : {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

Pytorch Version : 2.12.0+cu126
CUDA Available : True


In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Default Device : {device}")

Default Device : cuda


# 2. Load Dataset

In [9]:
XLSUM_DATASET_PATH = os.path.join("..", "data", "xlsum", "analysis_data.csv")
# TODO do parser for liputan6 dataset
# LIPUTAN6_DATASET_PATH =

class XLSumDataset(Dataset):
    """Custom Dataset For XLSum Only"""

    def __init__(self, df):
        self.df = df
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        return {
            # here id is use so we don't lose track the data and can be used
            # as name of the file if we want to store all the amr graph on .txt
            "id" : row["id"],
            "text" : row["text"],
        }

In [10]:
df = pd.read_csv(XLSUM_DATASET_PATH)
ds = XLSumDataset(df)

# 3. Load Tokenizer & Model

In [19]:
from transformers import NllbTokenizer, AutoModelForSeq2SeqLM
from huggingface_hub import snapshot_download

# Download model locally first to bypass transformers/huggingface_hub version mismatch
nllb_local_path = snapshot_download("facebook/nllb-200-distilled-1.3B")
print(f"NLLB model downloaded to: {nllb_local_path}")

nllb_tokenizer = NllbTokenizer.from_pretrained(nllb_local_path)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_local_path).to(device)
print(f"NLLB model loaded successfully!")

print(f"Model Loaded on : {nllb_model.device}")
print(f"Model Parameters : {sum([p.numel() for p in nllb_model.parameters() if p.requires_grad is True])}")

Fetching 9 files: 100%|██████████| 9/9 [00:00<?, ?it/s]


NLLB model downloaded to: C:\Users\Lenovo\.cache\huggingface\hub\models--facebook--nllb-200-distilled-1.3B\snapshots\7be3e24664b38ce1cac29b8aeed6911aa0cf0576
NLLB model loaded successfully!
Model Loaded on : cuda:0
Model Parameters : 1370638336


# 4. Create Data Loader

In [12]:
dl = DataLoader(ds, batch_size=4)

In [ ]:
def save_file(folder_path : str):
    os.makedirs(folder_path, exist_ok=True)

    def process(filename : str, translated : str):
        with open(os.path.join(folder_path, filename), "w") as f:
            f.write(translated)

    def inner_fn(filename : str, translated : str):
        try:
            process(filename, translated)
            print(f"Successfully write translatd file for {filename}")
        except Exception as e:
            print("Something went wrong")
            print(e)

    return inner_fn

# 5. Translate

In [ ]:
OUTPUT_PATH = "output/xlsum/translate"
save_file_fn = save_file(OUTPUT_PATH)
for batch in dl:
    ids, texts = batch["id"], batch["text"]
    
    tokenized_texts = nllb_tokenizer(
        texts, padding = True, truncation = True, max_length = None,
        return_tensors="pt"
    ).to(nllb_model.device)
    
    translated_tokens = nllb_model.generate(
        input_ids = tokenized_texts["input_ids"],
        attention_mask = tokenized_texts["attention_mask"],
        forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids("eng_Latn")
    )

    translated_texts = nllb_tokenizer.batch_decode(translated_tokens,
                                                    skip_special_tokens=True)
    
    for id, text in zip(ids, texts):
        save_file_fn(f"{id}.txt", text)

c:\D\ITB\riset_gnn\generate_amr\venv\lib\site-packages\transformers\generation_utils.py:1202: UserWarning: Neither `max_length` nor `max_new_tokens` have been set, `max_length` will default to 200 (`self.config.max_length`). Controlling `max_length` via the config is deprecated and `max_length` will be removed from the config in v5 of Transformers -- we recommend using `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


tensor([[     2, 256047,   1617,   7826,    452,    349,    362,  19199,   1482,
           1398,    216,  30382,    796,  38406, 180436,    281,   9904,  13894,
         248075,   1617,    362,  19199,   8488,   8910,    108,      9, 105817,
            159, 144483, 248079,   4244, 183968,    540,    337,  30382,   9949,
            452,    349, 111487,    452,    349,  76729,   2510, 248075,   1617,
          34558,    452,    349,    362,  19199, 248116, 248066, 157466,    216,
          30382,    248,   2294,   5563,  53693, 248075,   1617,    362,  19199,
           1398,   4719,  46163,    811,  21347,  27523,    349, 111487,    540,
         241260, 248075, 156260,  13894, 125172,   1482,  28911,    362,  19199,
           7131,   9903,    216,  46163,    796,   5057,  30158,  44388, 248075,
              2,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,    

In [25]:
nllb_tokenizer.batch_decode(translated_texts, skip_special_tokens=True)

["The video of the gajah that was mucking it went viral on social media. The gajah ran off in a direction unknown, crashing and kicking some of the participants of the ceremony. The cause of the gajah's sudden mucking is not yet known. The gajah was shocked by something among the participants and visitors. Local media reported that another gajah who also mucked it from different countries.",
 "The project is undergoing a detailed evaluation, but the East-West corridor is being withdrawn from the PSN. The construction of the project is being considered as the prime target of the government of President Joko Widodo, with a target of 245 projects until the year 2019, but 14 of them have been removed from the list of National Strategic Projects due to the lack of financial resources. The project is expected to be completed in February 2019 and the project is expected to be completed in February. The project is expected to involve a number of private investments, including the construction 